In [ ]:
# =========================================
# Create a customer-level LTV dashboard by cleaning data,
#excluding test users, aggregating purchase metrics (overall, last 12M, last FY),
#segmenting customers into LTV buckets, and generating a summary report
# =========================================
from google.colab import files
uploaded = files.upload()

# =========================================
# STEP 2: Load & Clean Data
# =========================================
import pandas as pd
import numpy as np

file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name, engine='openpyxl')

# -------- CLEANING --------
df['identity'] = (
    df['identity']
    .astype(str)
    .str.replace('.0','', regex=False)
    .str.strip()
)

df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

df['Financial Year Flag'] = df['Financial Year Flag'].astype(str).str.strip()
df['Last 12 Month Purchased'] = df['Last 12 Month Purchased'].astype(str).str.strip()

df['Total Billing Value'] = pd.to_numeric(df['Total Billing Value'], errors='coerce').fillna(0)

# =========================================
# STEP 3: EXCLUDE IDENTITIES
# =========================================
exclude_ids = [
    "1111119999999",
    "1119999999",
    "9999988800",
    "353777693"
]

df = df[~df['identity'].isin(exclude_ids)]

print("After exclusion - rows:", len(df))
print("Unique customers:", df['identity'].nunique())

# =========================================
# STEP 4: CUSTOMER MASTER
# =========================================
customer = df.groupby('identity', as_index=False).agg(
    Total_LTV=('Total Billing Value', 'sum'),
    Total_Purchases=('identity', 'count'),
    First_Purchase=('ts', 'min'),
    Last_Purchase=('ts', 'max')
)

# =========================================
# STEP 5: LAST 12M METRICS
# =========================================
df_12m = df[df['Last 12 Month Purchased'] == 'Last 12 Month']

cust_12m = df_12m.groupby('identity', as_index=False).agg(
    Last_12M_LTV=('Total Billing Value', 'sum'),
    Last_12M_Purchases=('identity', 'count')
)

customer = customer.merge(cust_12m, on='identity', how='left')

# =========================================
# STEP 6: LAST FY METRICS (FIXED)
# =========================================
FY_TARGET = "FY 2025-26"

df_fy = df[df['Financial Year Flag'] == FY_TARGET]

cust_fy = df_fy.groupby('identity', as_index=False).agg(
    Last_FY_Billing=('Total Billing Value', 'sum'),
    Last_FY_Purchases=('identity', 'count')
)

customer = customer.merge(cust_fy, on='identity', how='left')

# =========================================
# STEP 7: FILL NULLS
# =========================================
customer[['Last_12M_LTV','Last_12M_Purchases',
          'Last_FY_Billing','Last_FY_Purchases']] = \
customer[['Last_12M_LTV','Last_12M_Purchases',
          'Last_FY_Billing','Last_FY_Purchases']].fillna(0)

# =========================================
# STEP 8: DERIVED METRICS
# =========================================
customer['Avg_LTV'] = np.where(
    customer['Total_Purchases'] > 0,
    customer['Total_LTV'] / customer['Total_Purchases'],
    0
)

customer['Avg_Last_FY'] = np.where(
    customer['Last_FY_Purchases'] > 0,
    customer['Last_FY_Billing'] / customer['Last_FY_Purchases'],
    0
)

customer['Active_Flag'] = np.where(
    customer['Last_12M_LTV'] > 0,
    'Active',
    'Inactive'
)

# =========================================
# STEP 9: LTV BUCKETS
# =========================================
quantiles = {
    "Top 50%": 0.5,
    "Top 25%": 0.75,
    "Top 15%": 0.85,
    "Top 10%": 0.9,
    "Top 5%": 0.95,
    "Top 2.5%": 0.975,
    "Top 1%": 0.99
}

cutoffs = {k: customer['Total_LTV'].quantile(v) for k,v in quantiles.items()}

def assign_bucket(x):
    if x >= cutoffs["Top 1%"]:
        return "Top 1%"
    elif x >= cutoffs["Top 2.5%"]:
        return "Top 2.5%"
    elif x >= cutoffs["Top 5%"]:
        return "Top 5%"
    elif x >= cutoffs["Top 10%"]:
        return "Top 10%"
    elif x >= cutoffs["Top 15%"]:
        return "Top 15%"
    elif x >= cutoffs["Top 25%"]:
        return "Top 25%"
    elif x >= cutoffs["Top 50%"]:
        return "Top 50%"
    else:
        return "Bottom 50%"

customer['Bucket'] = customer['Total_LTV'].apply(assign_bucket)

# =========================================
# STEP 10: DASHBOARD SUMMARY
# =========================================
summary_rows = []

bucket_order = [
    "Total","Top 50%","Top 25%","Top 15%",
    "Top 10%","Top 5%","Top 2.5%","Top 1%"
]

for b in bucket_order:
    if b == "Total":
        subset = customer.copy()
    else:
        subset = customer[customer['Bucket'] == b]

    total_ltv = subset['Total_LTV'].sum()
    cx_count = len(subset)
    total_purch = subset['Total_Purchases'].sum()
    avg_ltv = total_ltv / total_purch if total_purch > 0 else 0

    last_fy_bill = subset['Last_FY_Billing'].sum()
    cx_last_year = (subset['Last_FY_Billing'] > 0).sum()
    purch_last_year = subset['Last_FY_Purchases'].sum()
    avg_last_year = last_fy_bill / purch_last_year if purch_last_year > 0 else 0

    summary_rows.append([
        b,
        total_ltv,
        cx_count,
        total_purch,
        avg_ltv,
        last_fy_bill,
        cx_last_year,
        purch_last_year,
        avg_last_year
    ])

summary = pd.DataFrame(summary_rows, columns=[
    "Brackets",
    "Total LTV Billing Value",
    "Count of Cx",
    "No of Purchased",
    "Average Sales Value LTV",
    "Last Year Billing Value",
    "Cx count Last Year",
    "No. of Purchased Last Year",
    "Average Sales Value Last Year"
])

# =========================================
# STEP 11: LTV %
# =========================================
total_ltv_all = summary.loc[summary['Brackets']=="Total","Total LTV Billing Value"].values[0]

summary['LTV Billing %'] = summary['Total LTV Billing Value'] / total_ltv_all

# =========================================
# STEP 12: EXPORT
# =========================================
with pd.ExcelWriter("Final_Dashboard.xlsx") as writer:
    customer.to_excel(writer, sheet_name="Customer_Master", index=False)
    summary.to_excel(writer, sheet_name="Dashboard", index=False)

print("✅ Final Dashboard Created Successfully!")